# GroundedX / RAN-Doc revision runs (Kaggle, 2× T4)

**Settings → Accelerator: GPU T4 ×2, Internet: ON.** Then *Save Version → Save & Run All (Commit)*.

The notebook is resumable. Each session runs until `DEADLINE_HOURS`, then zips `results/v2` to
`/kaggle/working/results_v2.zip`. For the next session, add the previous version's output as an
input (*Add Input → Your Work → this notebook*) and run again. Finished seeds are kept and unfinished
ones continue.

Set `PHASES` in the first code cell: `"pilot"` = hybrid validation pilot (~1 h incl. setup); `"hybrid"` = full test runs of the hybrid system + fault-reference ablation (~4 h). Earlier phases are already in the repo and are skipped automatically.

Order: accuracy jobs (E1/E3 first, then the E2 sweep, then E7 k=1/3). Profiling runs after that
(`RUN_PROFILING`) and is tagged `t4`. Edge-GPU (RTX 3050) profiling is a separate local run.

In [ ]:
# ---- configuration ----
REPO_URL = "https://github.com/TanishqX-404/groundedx.git"
BRANCH = "revision-minds"
DEADLINE_HOURS = 11.0      # Kaggle hard limit is 12 h; leaves time to zip outputs
RUN_ACCURACY = True
PHASES = ["pilot"]          # then ["hybrid"] after the pilot has been reviewed
RUN_PROFILING = True       # runs only after all accuracy jobs are finished
GPUS = ["0", "1"]

In [ ]:
import glob, os, subprocess, sys, time, zipfile, shutil
T0 = time.time()

def run(args, check=True):
    """Run a command, stream its output into the cell, and stop the notebook on failure."""
    print("+", " ".join(map(str, args)), flush=True)
    proc = subprocess.Popen([str(a) for a in args], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="", flush=True)
    code = proc.wait()
    if check and code != 0:
        raise RuntimeError(f"command failed with exit code {code}")
    return code
!nvidia-smi
WORK = "/kaggle/working"
REPO = f"{WORK}/groundedx"
# Code: a Kaggle dataset containing the repo (if attached) wins, otherwise clone the branch.
src = [p for p in glob.glob("/kaggle/input/*/groundedx*") if os.path.isdir(p) and os.path.exists(f"{p}/pyproject.toml")]
if os.path.exists(REPO):
    shutil.rmtree(REPO)
if src:
    shutil.copytree(src[0], REPO)
    print("code from dataset", src[0])
else:
    subprocess.check_call(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, REPO])
# Restore results from a previous session (output of an earlier version attached as input).
for z in glob.glob("/kaggle/input/**/results_v2.zip", recursive=True):
    print("restoring", z)
    with zipfile.ZipFile(z) as f:
        f.extractall(f"{REPO}/results")
# Reuse the compiled llama-server and downloaded GGUFs from an attached earlier output, if any.
for b in glob.glob("/kaggle/input/**/llama-bin/llama-server", recursive=True)[:1]:
    os.makedirs(f"{WORK}/llama-bin", exist_ok=True)
    shutil.copy(b, f"{WORK}/llama-bin/llama-server"); os.chmod(f"{WORK}/llama-bin/llama-server", 0o755)
    print("reusing llama-server from", b)
os.makedirs(f"{REPO}/models", exist_ok=True)
for g in glob.glob("/kaggle/input/**/models/*.gguf", recursive=True):
    dst = f"{REPO}/models/{os.path.basename(g)}"
    if not os.path.exists(dst):
        os.symlink(g, dst); print("reusing", g)
os.chdir(REPO)
!git log --oneline -1 || true

In [ ]:
# Build llama.cpp's llama-server with CUDA at the pinned build used for all revision runs
# (~10-15 min; static binary, T4 = sm_75). Skipped if a binary already exists.
LLAMA_TAG = "b11398"
BIN = f"{WORK}/llama-bin/llama-server"

def sh(cmd):
    print("+", cmd[:200], flush=True)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if r.returncode != 0:
        print(r.stdout[-3000:], r.stderr[-3000:])
    return r.returncode == 0

if not os.path.exists(BIN):
    nvcc = shutil.which("nvcc") or next(iter(glob.glob("/usr/local/cuda*/bin/nvcc")), None)
    assert nvcc, "nvcc not found - is the GPU accelerator enabled?"
    cuda_root = os.path.dirname(os.path.dirname(os.path.realpath(nvcc)))
    # libcuda: prefer the real driver library mounted by the GPU runtime, else the toolkit stub.
    libcuda = next(iter(
        glob.glob("/usr/lib/x86_64-linux-gnu/libcuda.so*") + glob.glob("/usr/lib64/libcuda.so*")
        + glob.glob(f"{cuda_root}/lib64/stubs/libcuda.so") + glob.glob(f"{cuda_root}/targets/*/lib/stubs/libcuda.so")
    ), None)
    print("nvcc:", nvcc, "| cuda root:", cuda_root, "| libcuda:", libcuda)
    sh(f"rm -rf /tmp/llama.cpp && git -c advice.detachedHead=false clone -q --depth 1 -b {LLAMA_TAG} https://github.com/ggml-org/llama.cpp /tmp/llama.cpp")
    base = (f"cd /tmp/llama.cpp && rm -rf build && cmake -B build -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=75 "
            f"-DCMAKE_CUDA_COMPILER={nvcc} -DCUDAToolkit_ROOT={cuda_root} "
            "-DBUILD_SHARED_LIBS=OFF -DLLAMA_CURL=OFF -DLLAMA_BUILD_TESTS=OFF -DLLAMA_BUILD_EXAMPLES=OFF")
    attempts = []
    if libcuda:
        attempts.append(("driver-lib", f"{base} -DCUDA_cuda_driver_LIBRARY={libcuda}"))
    # Fallback: legacy CUDA memory pool, which does not link libcuda directly.
    attempts.append(("no-vmm", f"{base} -DGGML_CUDA_NO_VMM=ON"))
    for name, cfg_cmd in attempts:
        if sh(cfg_cmd) and sh("cd /tmp/llama.cpp && cmake --build build --config Release -j4 --target llama-server"):
            print("build OK with", name)
            open(f"{WORK}/llama_build_variant.txt", "w").write(name)
            break
    else:
        raise RuntimeError("llama-server build failed with all configurations (see output above)")
    os.makedirs(f"{WORK}/llama-bin", exist_ok=True)
    shutil.copy("/tmp/llama.cpp/build/bin/llama-server", BIN)
os.environ["LLAMA_SERVER_BIN"] = BIN
print(subprocess.run([BIN, "--version"], capture_output=True, text=True).stderr)
!pip install -q -e . nvidia-ml-py xgboost huggingface_hub pytest

In [ ]:
import yaml
from huggingface_hub import hf_hub_download
cfg = yaml.safe_load(open("configs/experiment.yaml"))
os.makedirs("models", exist_ok=True)
for key, m in cfg["generation"]["models"].items():
    print(key, hf_hub_download(m["repo"], m["file"], local_dir="models"))
!ls -la models

In [ ]:
# Correctness gate (claims tests C1-C8) incl. a real constrained decode through llama-server on GPU.
os.environ["GROUNDEDX_TEST_MODEL"] = "models/qwen2.5-1.5b-instruct-q4_k_m.gguf"
run([sys.executable, "-m", "pytest", "-q", "-x", "tests"])

In [ ]:
# Speed probe: 10 windows of the main config (separate experiment name, not used in results).
run([sys.executable, "scripts/run_llm.py", "--model", "qwen2.5-3b-q4", "--model-path", "models/qwen2.5-3b-instruct-q4_k_m.gguf",
     "--mode", "rag", "--seeds", "7", "--limit", "10", "--experiment", "_speedprobe"])
import json, statistics
rows = [json.loads(l) for l in open("results/v2/_speedprobe/seed_7/predictions.jsonl")]
print("median s/call:", statistics.median(r["latency_ms"] for r in rows) / 1000, "valid:", sum(r["valid"] for r in rows), "/", len(rows))
print("example output:", rows[0]["raw"][:400])
shutil.rmtree("results/v2/_speedprobe")

In [ ]:
for phase in (["accuracy"] if RUN_ACCURACY else []) + PHASES:
    remaining = DEADLINE_HOURS - (time.time() - T0) / 3600
    run([sys.executable, "scripts/run_queue.py", "--phase", phase, "--gpus", *GPUS,
         "--models-dir", "models", "--deadline-hours", f"{remaining:.2f}"], check=False)
run([sys.executable, "scripts/make_results.py"], check=False)
# Print the hybrid / pilot sections so the decision can be read straight from the log.
text = open("results/v2/RESULTS.md").read()
for header in ("## Hybrid", "## Validation-split pilots"):
    if header in text:
        print(text[text.index(header):].split(chr(10) + "## ")[0])

In [ ]:
import importlib.util
spec = importlib.util.spec_from_file_location("rq", "scripts/run_queue.py"); rq = importlib.util.module_from_spec(spec); spec.loader.exec_module(rq)
left = [(ph, j) for ph in ["accuracy"] + PHASES for j in rq.PHASE_JOBS[ph] if not rq.accuracy_done(j)]
print("unfinished accuracy/pilot/hybrid jobs:", len(left))
for ph, j in left:
    print("  not finished:", ph, j)
remaining = DEADLINE_HOURS - (time.time() - T0) / 3600
# Profiling (skips configs already profiled) only once every requested phase is complete.
if RUN_PROFILING and not left and "hybrid" in PHASES and remaining > 0.3:
    # one GPU at a time so the two profiles do not contend for CPU
    run([sys.executable, "scripts/run_queue.py", "--phase", "profile", "--gpus", "0",
         "--models-dir", "models", "--tag", "t4", "--deadline-hours", f"{remaining:.2f}"])

In [ ]:
run([sys.executable, "scripts/make_results.py"])
with zipfile.ZipFile(f"{WORK}/results_v2.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for path in glob.glob("results/v2/**", recursive=True):
        if os.path.isfile(path):
            z.write(path, os.path.relpath(path, "results"))
print("zipped ->", f"{WORK}/results_v2.zip", os.path.getsize(f"{WORK}/results_v2.zip") // 1024, "KB")
print(open("results/v2/RESULTS.md").read()[:6000])